# E4 / RQ3 — Analysis · Reuse E3B/E3C · STRICT QA
**Group 14 | Colab | Read-only source inputs | 18 runs × K={5,10,20}**

Tái sử dụng công thức accuracy và popularity bias từ `12_E3C_Analysis_Weight_Sensitivity.ipynb`, QA Top-K từ E3C, paired depth/model comparisons từ bản E4 SAFE và ý tưởng biểu đồ trade-off của `10_E3B_Popularity_Bias_Analysis.ipynb`.

**Chỉ chạy sau khi E4 Export Top-K FIXED hoàn tất 18/18.** Không train, không chạy inference, không sửa checkpoint, không ghi đè exports. Nếu bất kỳ run nào thiếu hoặc QA fail, notebook dừng trước khi xuất báo cáo.

**Protocol:** test users có ít nhất một test positive; chỉ mask train; Head=ceil(20% item) theo binary train degree, hòa thì item ID tăng dần; Tail=phần còn lại; macro Recall/NDCG; conditional macro Head/TailRecall; Coverage và Spearman tính trên toàn catalog. K=5,10,20; 3 seeds; sample SD (ddof=1).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, pickle, hashlib, math
from datetime import datetime, timezone
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA=ROOT/'data/raw/amazon'; E4=ROOT/'experiments/E4'
EXPORT=E4/'exports'; MODELS=('lightgcn','lightgcn_plus'); LAYERS=(1,2,3); SEEDS=(2023,2024,2025)
KS=(5,10,20); N_USERS=11000; N_ITEMS=9332; N_TEST_USERS=9860
METRICS=['Recall','NDCG','ARP','APLT','HeadRecall','TailRecall','Coverage','DegreeExposureSpearman']
assert DATA.is_dir() and EXPORT.is_dir(), 'Missing Drive dataset or E4/exports'
print('Read-only evaluation |',EXPORT)

In [ ]:
with (DATA/'trn_mat.pkl').open('rb') as f: train=pickle.load(f)
with (DATA/'tst_mat.pkl').open('rb') as f: test=pickle.load(f)
assert sp.issparse(train) and sp.issparse(test)
train=train.tocsr(copy=True);test=test.tocsr(copy=True)
for x in (train,test):
    x.sum_duplicates();x.eliminate_zeros();x.sort_indices();x.data[:]=1
assert train.shape==test.shape==(N_USERS,N_ITEMS)
users_expected=np.flatnonzero(np.diff(test.indptr)>0)
assert len(users_expected)==N_TEST_USERS
item_degree=np.asarray(train.getnnz(axis=0)).ravel()
head_count=math.ceil(.2*N_ITEMS)
head_order=np.lexsort((np.arange(N_ITEMS),-item_degree))
head_mask=np.zeros(N_ITEMS,dtype=bool);head_mask[head_order[:head_count]]=True
print('Dataset verified | test users',len(users_expected),'| head items',head_count)


In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()

def locate(model,layer,seed):
    return EXPORT/model/f'layers_{layer}'/f'seed_{seed}'/'topk'/'topk_test.npz'

inventory=[]; exports={}; trainer_refs={}; failures=[]
for model in MODELS:
 for layer in LAYERS:
  for seed in SEEDS:
   p=locate(model,layer,seed)
   rec=dict(model=model,layer=layer,seed=seed,path=str(p),status='MISSING',sha256='')
   try:
    assert p.is_file(), 'Top-K file missing'
    provenance=p.parent/'E4_export_provenance.json'
    assert provenance.is_file(), 'Missing E4 provenance'
    meta=json.loads(provenance.read_text(encoding='utf-8'))
    assert meta.get('status')=='passed'
    assert (meta.get('model'),int(meta.get('layer')),int(meta.get('seed')))==(model,layer,seed)
    assert meta.get('ranking_mask')=='train_only' and meta.get('validation_masked') is False
    assert isinstance(meta.get('trainer_metrics'),dict), 'Missing E4 trainer metrics'
    assert isinstance(meta.get('export_vs_trainer_abs_diff'),dict), 'Missing export/trainer QA'
    cp=Path(meta['checkpoint']);assert cp.is_file(), 'Source checkpoint not found'
    assert sha256(cp)==meta['checkpoint_sha256'], 'Checkpoint hash mismatch'
    with np.load(p,allow_pickle=False) as z:
        assert {'user_ids','item_ids','scores'}<=set(z.files)
        users=z['user_ids'];items=z['item_ids'];scores=z['scores']
    assert users.dtype.kind in 'iu' and items.dtype.kind in 'iu'
    assert users.shape==(N_TEST_USERS,) and items.shape==scores.shape==(N_TEST_USERS,20)
    assert np.array_equal(users,users_expected),'Test users/order mismatch'
    assert np.all((items>=0)&(items<N_ITEMS))
    assert np.isfinite(scores).all() and np.all(scores[:,:-1]>=scores[:,1:]-1e-6)
    assert np.all(np.diff(np.sort(items,axis=1),axis=1)!=0),'Duplicate recommendations'
    assert not np.asarray(train[np.repeat(users,20),items.ravel()]).any(),'TRAIN MASK LEAKAGE'
    rec.update(status='PASS',sha256=sha256(p))
    trainer_refs[(model,layer,seed)]=meta['trainer_metrics']
    exports[(model,layer,seed)]=items
   except Exception as exc:
    rec['status']='FAIL: '+str(exc)
    failures.append(rec)
   inventory.append(rec)
inventory_df=pd.DataFrame(inventory)
print(inventory_df[['model','layer','seed','status']].to_string(index=False))
assert len(exports)==18 and not failures, f'NOT READY: {len(exports)}/18 valid. Fix export before analysis.'
print('QA PASS 18/18')

In [ ]:
def evaluate(items,k):
    rec=items[:,:k];u=users_expected;truth=test[u]
    truth_count=np.asarray(truth.getnnz(axis=1)).ravel()
    hits=np.asarray(test[np.repeat(u,k),rec.ravel()]).reshape(len(u),k).astype(bool)
    hit_count=hits.sum(axis=1)
    recall=float(np.mean(hit_count/truth_count))
    discounts=1/np.log2(np.arange(2,k+2))
    dcg=hits@discounts
    idcg=np.cumsum(discounts)[np.minimum(truth_count,k)-1]
    ndcg=float(np.mean(dcg/idcg))
    head_truth=np.asarray(truth[:,np.flatnonzero(head_mask)].getnnz(axis=1)).ravel()
    tail_truth=truth_count-head_truth
    head_hits=(hits & head_mask[rec]).sum(axis=1)
    tail_hits=(hits & ~head_mask[rec]).sum(axis=1)
    assert np.array_equal(head_hits+tail_hits,hit_count)
    hv=head_truth>0;tv=tail_truth>0
    exposure=np.bincount(rec.ravel(),minlength=N_ITEMS)
    rho=float(spearmanr(item_degree,exposure).statistic)
    return dict(K=k,Recall=recall,NDCG=ndcg,
        ARP=float(item_degree[rec].mean()),APLT=float((~head_mask[rec]).mean()),
        HeadRecall=float(np.mean(head_hits[hv]/head_truth[hv])),
        TailRecall=float(np.mean(tail_hits[tv]/tail_truth[tv])),
        Coverage=float(np.count_nonzero(exposure)/N_ITEMS),DegreeExposureSpearman=rho,
        n_head_eligible=int(hv.sum()),n_tail_eligible=int(tv.sum()))

rows=[]
TRAINER_TOL=2e-4
for (model,layer,seed),items in exports.items():
    for k in KS:
        result=evaluate(items,k)
        trainer=trainer_refs[(model,layer,seed)]
        for key,metric in ((f'recall@{k}','Recall'),(f'ndcg@{k}','NDCG')):
            assert key in trainer, f'Missing trainer {key} for {(model,layer,seed)}'
            delta=abs(result[metric]-float(trainer[key]))
            assert delta<=TRAINER_TOL, f'Accuracy mismatch {(model,layer,seed)} {key}: computed={result[metric]:.8f}, trainer={trainer[key]}, diff={delta:.8f}'
        rows.append(dict(model=model,layer=layer,seed=seed,**result))
run_df=pd.DataFrame(rows).sort_values(['model','layer','seed','K']).reset_index(drop=True)
assert len(run_df)==54 and np.isfinite(run_df[METRICS].to_numpy()).all()
summary=run_df.groupby(['model','layer','K'],as_index=False).agg(**{
    f'{m}_{stat}':pd.NamedAgg(column=m,aggfunc=func)
    for m in METRICS for stat,func in [('mean','mean'),('sd',lambda x:x.std(ddof=1))]
})
summary['n_seeds']=3
print(summary[['model','layer','K','Recall_mean','Recall_sd','NDCG_mean','ARP_mean','APLT_mean','TailRecall_mean']].to_string(index=False))

In [ ]:
# Paired RLMRec-Con - LightGCN at SAME layer, seed, K.
a=run_df[run_df.model=='lightgcn'].set_index(['layer','seed','K'])
b=run_df[run_df.model=='lightgcn_plus'].set_index(['layer','seed','K'])
assert a.index.equals(b.index)
paired=b[['model']].drop(columns=['model'])
for m in METRICS:paired['delta_'+m]=b[m]-a[m]
paired=paired.reset_index()
paired_summary=paired.groupby(['layer','K'],as_index=False).agg(**{
    f'delta_{m}_{stat}':pd.NamedAgg(column='delta_'+m,aggfunc=func)
    for m in METRICS for stat,func in [('mean','mean'),('sd',lambda x:x.std(ddof=1))]
})
# Within-model depth comparisons: L2-L1, L3-L1, L3-L2, paired by seed.
depth_rows=[]
for model in MODELS:
 for k in KS:
  sub=run_df[(run_df.model==model)&(run_df.K==k)].set_index(['layer','seed'])
  for lo,hi in ((1,2),(1,3),(2,3)):
   for seed in SEEDS:
    row=dict(model=model,K=k,seed=seed,from_layer=lo,to_layer=hi)
    for m in METRICS:row['delta_'+m]=float(sub.loc[(hi,seed),m]-sub.loc[(lo,seed),m])
    depth_rows.append(row)
depth_df=pd.DataFrame(depth_rows)
print('Paired comparisons ready (descriptive, NOT significance tests)')


In [ ]:
# Save ONLY NEW timestamped analysis folder; never overwrite existing outputs.
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out=E4/'analysis'/stamp
out.mkdir(parents=True,exist_ok=False)
inventory_df.to_csv(out/'E4_topk_inventory_QA.csv',index=False)
run_df.to_csv(out/'E4_metrics_per_run.csv',index=False)
summary.to_csv(out/'E4_summary_mean_sd.csv',index=False)
paired.to_csv(out/'E4_paired_model_deltas.csv',index=False)
paired_summary.to_csv(out/'E4_paired_model_summary.csv',index=False)
depth_df.to_csv(out/'E4_paired_depth_deltas.csv',index=False)

for k in KS:
 for metric in ('Recall','NDCG','ARP','APLT','TailRecall','Coverage','DegreeExposureSpearman'):
  fig,ax=plt.subplots(figsize=(7,4.3))
  for model in MODELS:
   s=summary[(summary.model==model)&(summary.K==k)].sort_values('layer')
   ax.errorbar(s.layer,s[f'{metric}_mean'],yerr=s[f'{metric}_sd'],marker='o',capsize=4,label=model)
  ax.set_xticks(LAYERS);ax.set_xlabel('Graph propagation layers')
  ax.set_ylabel(f'{metric}@{k}');ax.set_title(f'E4 RQ3: {metric}@{k} (mean ± SD, n=3)')
  ax.grid(alpha=.2);ax.legend();fig.tight_layout()
  fig.savefig(out/f'E4_{metric}_at_{k}.png',dpi=170)
  plt.close(fig)

manifest={'status':'QA_PASS_18_OF_18','created_utc':datetime.now(timezone.utc).isoformat(),
 'protocol':'train-only masking; validation unmasked; test users with >=1 test item',
 'head_definition':'top ceil(20%) items by train degree, ties item ID ascending',
 'n_runs':18,'n_metric_rows':54,'source_exports':str(EXPORT),
 'trainer_accuracy_crosscheck_tolerance':TRAINER_TOL, 'notes':['Mean ± sample SD across 3 seeds','Paired comparisons descriptive only',
          'APLT is exposure, not relevance; assess with TailRecall','Spearman and ARP capture different bias dimensions']}
(out/'E4_analysis_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
# E3B-style accuracy–popularity trade-off plots; layer points connected per model.
for k in KS:
 for xmetric in ('ARP','APLT'):
  fig,ax=plt.subplots(figsize=(7.5,5))
  for model in MODELS:
   ss=summary[(summary.model==model)&(summary.K==k)].sort_values('layer')
   ax.errorbar(ss[f'{xmetric}_mean'],ss['Recall_mean'],
               xerr=ss[f'{xmetric}_sd'],yerr=ss['Recall_sd'],
               marker='o',capsize=3,label=model)
   for _,r in ss.iterrows():
    ax.annotate(f"L{int(r['layer'])}",(r[f'{xmetric}_mean'],r['Recall_mean']),
                xytext=(4,5),textcoords='offset points',fontsize=8)
  ax.set_xlabel(f'{xmetric}@{k}')
  ax.set_ylabel(f'Recall@{k}')
  ax.set_title(f'E4 RQ3: Accuracy–Popularity Trade-off @ {k}')
  ax.grid(alpha=.2);ax.legend();fig.tight_layout()
  fig.savefig(out/f'E4_Recall_vs_{xmetric}_at_{k}.png',dpi=180)
  plt.close(fig)

report=['E4 RQ3 — GRAPH DEPTH ANALYSIS','QA PASS 18/18; 54 run × K rows',
        'Accuracy cross-checked with E4 trainer metrics, tolerance '+str(TRAINER_TOL),
        'Model labels: lightgcn=LightGCN; lightgcn_plus=RLMRec-Con','']
for _,r in summary[summary.K==20].iterrows():
 report.append(f"{r['model']} L{int(r['layer'])}: Recall@20={r['Recall_mean']:.6f} ± {r['Recall_sd']:.6f}; NDCG@20={r['NDCG_mean']:.6f}; ARP@20={r['ARP_mean']:.4f}; APLT@20={r['APLT_mean']:.4f}; TailRecall@20={r['TailRecall_mean']:.6f}")
report+=['','Paired differences are descriptive only, not statistical significance.',
         'Interpret accuracy together with ARP, APLT and TailRecall.']
(out/'E4_RQ3_summary.txt').write_text('\\n'.join(report)+'\\n',encoding='utf-8')
print('DONE:',out)
print('Files:',*[p.name for p in sorted(out.iterdir())],sep='\n - ')

## Bàn giao cho người viết Midterm 2

Chỉ sử dụng kết quả khi cell QA báo `QA PASS 18/18`. Gửi nguyên thư mục `E4/analysis/<timestamp>` cùng `E4/exports/E4_export_status.csv`, các config/log training, và báo cáo E1/E2/E3A/E3B/E3C. Người viết cần đối chiếu các biểu đồ và paired deltas với RQ3/H3; tránh diễn giải ARP giảm hoặc APLT tăng thành giảm thiên lệch toàn diện. Báo cáo cần phân biệt dữ liệu cũ (18 training logs) và các run recovery mới có checkpoint; không nói là cùng một lần chạy nếu khác run.
